# BiGRU OOF export — `bigru_oof.pkl`

**Run these cells inside `rogii-seq-unet-gpu` (the BiGRU notebook), not the v22 fork.**

## Why this exists

`oof_mae.npy` is not per-well. Cell 10 does `for i in va: oof_mae[i] = best` — it
writes each fold's *mean* to every well in that fold, so it holds five distinct
values broadcast across 773 slots. It looks per-well and isn't. The join needs
real per-well errors and per-station predictions, which is what this produces.

## Two paths in

- **Path A (preferred, ~15 min).** Reload the trained folds from `seq_model.pt` in
  the v4 Output tab. No retrain, no GPU time beyond inference. R1 does this.
- **Path B (~70 min).** Retrain at **original v4 settings**. Use this only if
  `seq_model.pt` is missing.

If you still have the A–F edits (`win_max=12288`, GRU packing, EMA) in the
notebook, **revert to v4 settings first**. That configuration is what ran 11+
hours against the 12-hour cap and got killed, and the saved weights are v4's.

## Run order

Run **CELL 1 → 9** of the BiGRU notebook first (defines `BiGRUNet`, `make_batch`,
`assemble`, and builds `SEQS`/`METAS`; ~7 min). **Skip Cell 10** on Path A. Then
run R0 → R2 below.

## The gate that decides whether the output is usable

R2 must print an OOF near **6.922**. Materially *lower* is the dangerous
outcome — it means fold assignment drifted from what the checkpoint was trained
on, so wells are being scored by models that trained on them. That's a leak, and
it would flatter the BiGRU throughout the join.

In [ ]:
# ===== R0: preflight + signature probe =====
# The notebook may be in v4 shape or carry the A-F edits (make_batch returning 5
# values, forward taking `lens`). Probe rather than assume — guessing wrong here
# fails silently in the worst case.
import os, glob, pickle, inspect, time
import numpy as np, torch

_NEED = ['BiGRUNet', 'make_batch', 'SEQS', 'METAS', 'CFG', 'DEVICE']
_missing = [n for n in _NEED if n not in globals()]
if _missing:
    raise NameError('run CELLS 1-9 of the BiGRU notebook first; missing: %s' % _missing)

_MB_N = len(inspect.signature(make_batch).parameters)
_FWD = inspect.signature(BiGRUNet.forward).parameters
_FWD_LENS = 'lens' in _FWD
print('make_batch params      : %d' % _MB_N)
print('forward accepts `lens` : %s' % _FWD_LENS)
print('SEQS: %d wells | CFG hidden=%s n_folds=%s'
      % (len(SEQS), CFG.get('hidden'), CFG.get('n_folds')))

def _unpack_batch(out):
    """make_batch returns 4 (v4) or 5 (A-F edits) values."""
    if len(out) == 5:
        return out
    X, T, Y, M = out
    return X, T, Y, M, None

def _forward(model, X, T, Ls):
    if _FWD_LENS and Ls is not None:
        return model(X, T, Ls)
    return model(X, T)

# locate a checkpoint for Path A
_CANDS = sorted(glob.glob('/kaggle/input/*/seq_model.pt')) + \
         sorted(glob.glob('seq_model.pt')) + \
         sorted(glob.glob('/kaggle/working/seq_model.pt'))
print('\ncheckpoint candidates:', _CANDS or 'NONE FOUND')
if not _CANDS:
    print('  -> Path A unavailable. Either add the v4 run as Notebook Output input')
    print('     (right panel -> Add Input -> Notebook Output -> rogii-seq-unet-gpu),')
    print('     or take Path B: run Cell 10 at ORIGINAL v4 settings, then skip R1.')
CKPT_PATH = _CANDS[0] if _CANDS else None


## R1 — rebuild `FOLD_MODELS` from the checkpoint (Path A)

Skips itself if `FOLD_MODELS` is already in memory from a fresh Cell 10 run, so
it's safe on both paths.

The checkpoint stores `ckpt['folds']` as a list of `(state_dict, mu, sd)`, one per
fold, **in fold order**. That ordering is load-bearing: fold `f`'s model must be
applied to fold `f`'s validation wells. Get it wrong and each model scores wells
it trained on.

In [ ]:
# ===== R1: reload trained folds (Path A) =====
if 'FOLD_MODELS' in globals() and FOLD_MODELS:
    print('FOLD_MODELS already in memory (%d folds) — skipping reload.' % len(FOLD_MODELS))
else:
    if CKPT_PATH is None:
        raise RuntimeError('no seq_model.pt — take Path B (run Cell 10 at v4 settings)')
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    if 'folds' not in ckpt:
        raise KeyError('checkpoint has no "folds" key; keys = %s' % list(ckpt))
    _nch = globals().get('N_CH'); _twd = globals().get('TW_DIM')
    if _nch is None or _twd is None:
        raise NameError('N_CH / TW_DIM not defined — re-run CELLS 1-9')
    # Take architecture from the CHECKPOINT's own cfg, not the live CFG. If the
    # notebook still carries the A-F edits, live CFG describes a model that was
    # never trained; load_state_dict would then fail loudly on a shape mismatch,
    # or worse, succeed if only non-shape params drifted.
    _ck = ckpt.get('cfg', {})
    _hidden = _ck.get('hidden', CFG['hidden'])
    FOLD_MODELS = []
    for (state, mu, sd) in ckpt['folds']:
        mdl = BiGRUNet(_nch, _twd, _hidden).to(DEVICE)
        mdl.load_state_dict(state); mdl.eval()
        FOLD_MODELS.append((mdl, mu, sd))
    print('reloaded %d fold models from %s (hidden=%s)'
          % (len(FOLD_MODELS), CKPT_PATH, _hidden))

    # win_max is load-bearing: the checkpoint trained at 4096, where every
    # sequence is exactly full so packing is a no-op. A longer win_max changes
    # SEQS assembly and silently mis-feeds these weights.
    if 'win_max' in _ck:
        print('ckpt win_max=%s | live CFG win_max=%s' % (_ck['win_max'], CFG['win_max']))
        assert _ck['win_max'] == CFG['win_max'], (
            'win_max mismatch — set CFG["win_max"]=%s and re-run CELLS 1-9 '
            '(SEQS must be rebuilt).' % _ck['win_max'])
    else:
        print('NOTE: checkpoint carries no cfg["win_max"]; confirm CFG win_max=4096')

assert len(FOLD_MODELS) == CFG['n_folds'], \
    'fold count mismatch: %d models vs n_folds=%d' % (len(FOLD_MODELS), CFG['n_folds'])


## R1b — recover the fold assignment

`folds` comes from `np.array_split(perm, CFG['n_folds'])` with a seeded
permutation, so re-running Cell 9 reproduces it. If `folds` survived in memory we
use it directly; otherwise we regenerate from the seed and **verify**.

The verification is the important part. Regenerating a permutation that doesn't
match what the checkpoint trained on produces no error — just quietly optimistic
numbers, because each model would be scoring wells it saw in training. R2's OOF
gate is the backstop.

In [ ]:
# ===== R1b: fold assignment =====
WELL_NAMES = [METAS[i]['well'] for i in range(len(SEQS))]

if 'folds' in globals() and folds is not None and len(folds) == CFG['n_folds']:
    FOLDS = [np.asarray(f) for f in folds]
    print('using `folds` from memory')
else:
    # Cell 10 uses the LEGACY global RNG. np.random.seed(0)+np.random.permutation
    # (MT19937) and np.random.default_rng(0).permutation (PCG64) produce entirely
    # different orderings — fold-0 overlap is ~18%, so most wells would be scored
    # by a model that trained on them. Reproduce Cell 10 exactly.
    np.random.seed(0)
    _perm = np.random.permutation(np.arange(len(SEQS)))
    FOLDS = [np.asarray(f) for f in np.array_split(_perm, CFG['n_folds'])]
    print('REGENERATED folds via np.random.seed(0) + np.random.permutation')
    print('  (this is Cell 10\'s exact recipe — the legacy global RNG, NOT default_rng)')
    print('  R2\'s OOF gate is the backstop if Cell 10 seeded differently.')

_cov = np.concatenate(FOLDS)
assert len(np.unique(_cov)) == len(SEQS), 'folds do not partition the wells'
print('fold sizes:', [len(f) for f in FOLDS], '| total', len(_cov))

pickle.dump({'folds': [list(map(int, f)) for f in FOLDS],
             'well_names': WELL_NAMES}, open('bigru_folds.pkl', 'wb'))
print('saved bigru_folds.pkl')


## R2 — per-well OOF predictions → `bigru_oof.pkl`

For each fold, runs that fold's model over its held-out wells and converts back to
**absolute TVT**: undo the increment with the well's `u_last`, add the datum, then
subtract `Z`.

That conversion is the whole ballgame for comparability. The BiGRU trains on
`u − u_lastknown`; v22 has its own anchor logic. The two are only comparable if
both errors are `abs(pred_TVT − true_TVT)` over the same blind stations of the
same wells. Everything here is pushed into absolute TVT space for that reason.

Stations are recorded with their **row indices**, so the join can intersect
exactly rather than assuming both models scored identical station sets.

In [ ]:
# ===== R2: per-well OOF predictions in absolute TVT space =====
bigru_err, bigru_pred = {}, {}
skipped = []
t0 = time.time()

for fi, va in enumerate(FOLDS):
    model, mu, sd = FOLD_MODELS[fi]
    model.eval()
    for j in va:
        j = int(j)
        w = WELL_NAMES[j]
        try:
            meta = METAS[j]
            z = np.asarray(meta['z'], dtype=float)
            with torch.no_grad():
                X, T, Y, M, Ls = _unpack_batch(
                    make_batch([j], SEQS, TGTS, MASKS, TWS, mu, sd))
                p = _forward(model, X, T, Ls)[0, :len(z)].cpu().numpy()

            # ERROR in increment space. |pred_incr - target_incr| == |TVT_hat - TVT|
            # because the per-well offset cancels. Doing it this way means the
            # error numbers stay correct even if the TVT reconstruction below is
            # wrong -- the two concerns are deliberately separated.
            m = np.asarray(MASKS[j], dtype=float)
            tgt = np.asarray(TGTS[j], dtype=float)
            L = min(len(p), len(tgt), len(m))
            if m[:L].sum() < 1:
                skipped.append((w, 'no blind stations in mask')); continue
            bigru_err[w] = float((np.abs(p[:L] - tgt[:L]) * m[:L]).sum() / m[:L].sum())

            # PREDICTIONS back to absolute TVT: increment + u_last - z.
            # No datum term -- it is already folded into u_last.
            tvt_hat = p.astype(float) + float(meta['u_last']) - z[:len(p)]
            bidx = np.where(m[:L] > 0)[0]
            bigru_pred[w] = dict(blind_idx=bidx.astype(np.int32),
                                 tvt_hat=tvt_hat[bidx].astype(np.float32),
                                 fold=int(fi))
        except Exception as e:
            skipped.append((w, repr(e)[:120]))
    print('fold %d done: %d wells, %.0fs' % (fi, len(va), time.time() - t0), flush=True)

pickle.dump({'err': bigru_err, 'pred': bigru_pred,
             'folds': {WELL_NAMES[int(j)]: fi for fi, va in enumerate(FOLDS) for j in va},
             'skipped': skipped}, open('bigru_oof.pkl', 'wb'))

_e = np.array(list(bigru_err.values()))
print('\n' + '=' * 62)
print('saved bigru_oof.pkl | OOF %.3f over %d wells' % (_e.mean(), len(_e)))
print('  p10 %.2f  median %.2f  p90 %.2f  max %.2f'
      % tuple(np.percentile(_e, [10, 50, 90]).tolist() + [_e.max()]))
if skipped:
    print('  skipped %d: %s' % (len(skipped), skipped[:5]))

# The cheapest, most decisive check: oof_mae.npy held five fold means broadcast
# across wells. If this file has ~5 distinct values it is the same useless
# artifact wearing a different name.
_nuniq = len(np.unique(np.round(_e, 4)))
print('  distinct error values: %d  [GATE: must be in the hundreds, NOT ~5]' % _nuniq)
if _nuniq <= 10:
    print('  *** FAILED — this is fold means, not per-well errors. Do not use it. ***')
print('-' * 62)
if 6.3 <= _e.mean() <= 7.6:
    print('GATE PASSED — %.3f is consistent with the v4 run\'s 6.922.' % _e.mean())
elif _e.mean() < 6.3:
    print('GATE FAILED (%.3f too LOW) — the dangerous failure. Fold assignment' % _e.mean())
    print('  likely drifted, so models are scoring wells they trained on.')
    print('  Re-run Cell 9 to restore the exact split, then re-run R1b and R2.')
else:
    print('GATE FAILED (%.3f too HIGH) — check the u_last/datum reconstruction' % _e.mean())
    print('  in R2 matches your METAS keys, and that v4 settings are restored.')
print('=' * 62)
print('\nDownload bigru_oof.pkl AND bigru_folds.pkl from the Output tab.')


## Notes

**If `METAS` doesn't carry `u_last` / `datum` under those names**, R2's
reconstruction is wrong and the OOF gate will fire high. Print
`METAS[0].keys()` and adjust the two `meta.get(...)` calls — the structure is
correct, only the key names would be off.

**Path B.** If `seq_model.pt` is missing, run Cell 10 at original v4 settings
(`epochs=250`, `patience=60`, `win_max=4096`, no packing), then skip R1 and run
R1b → R2. `FOLD_MODELS` will already be in memory.

**Don't chase v5.** The A–F edits were meant to expose the far toe and would have
raised OOF into a more honest 7.5–9, but that run died against the 12-hour cap.
The join works fine on v4's 6.922 — it just means both models are being scored on
the same partially-contaminated validation region, which is at least *symmetric*
and therefore still a fair comparison between them.